# Chapter 13 — Canvas, images, drawing, and visual annotations

**Book role:** Core GUI  
**Suggested book space:** about 7 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Use Canvas for drawing and custom visual objects.
- Load image assets with explicit path and reference management.
- Build simple annotations and lightweight sparklines.
- Choose Canvas versus a charting library deliberately.

## 1. Canvas as a coordinate-based surface

`Canvas` provides items such as lines, rectangles, ovals, text, polygons, and image objects. Every created item has an item ID and can receive tags that group behavior. Canvas is useful for custom diagrams, simple process flows, selection overlays, or a sparkline where a full plotting library would be excessive.

Coordinate-based drawing means you must define scaling behavior. Use the actual Canvas width and height, bind to a configure event when dimensions change, and redraw from model values rather than manually dragging every item into place. Avoid using Canvas to recreate complex tables or statistical charts; specialized widgets and Matplotlib provide better semantics for those tasks.

### Worked example 1: Draw a simple bar chart on Canvas

The example redraws from source values on resize. It demonstrates coordinates rather than best practices for statistical graphics; use Matplotlib for richer charts.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("Canvas bars")
root.geometry("520x300")
canvas = tk.Canvas(root, background="white", height=220)
canvas.pack(fill="both", expand=True, padx=12, pady=12)
values = [42, 65, 28, 83]
labels = ["A", "B", "C", "D"]


def draw(_event=None):
    canvas.delete("all")
    width = max(canvas.winfo_width(), 200)
    height = max(canvas.winfo_height(), 120)
    margin = 32
    usable_h = height - 2 * margin
    bar_w = (width - 2 * margin) / len(values) * 0.62
    gap = (width - 2 * margin) / len(values)
    maximum = max(values) or 1
    canvas.create_line(margin, height - margin, width - margin, height - margin, fill="#555555")
    for i, value in enumerate(values):
        x0 = margin + i * gap + (gap - bar_w) / 2
        y0 = height - margin - (value / maximum) * usable_h
        canvas.create_rectangle(x0, y0, x0 + bar_w, height - margin, fill="#4C78A8", outline="")
        canvas.create_text(x0 + bar_w / 2, y0 - 10, text=str(value))
        canvas.create_text(x0 + bar_w / 2, height - margin + 12, text=labels[i])

canvas.bind("<Configure>", draw)
root.mainloop()

## 2. Image lifetime and file paths

Tk images must retain a Python reference while they are displayed. A common failure is creating `PhotoImage` in a short-lived local variable and watching the image disappear when garbage collection releases it. Store image objects on the application or view instance. Use pathlib to locate packaged assets and document supported image formats; formats vary by Tk build and image handler availability.

When loading user-selected assets, validate that the file exists and can be opened, and handle errors gracefully. Large images should be resized for the target widget before loading when practical. For report imagery, include an alternative text label or nearby caption so meaning is not conveyed only through pixels.

### Worked example 2: Load an image and keep a reference

Do not expect this example to run unless the supplied path exists and the Tk build supports the image format. For JPEG and more formats, Pillow (`PIL.ImageTk`) is commonly used as an optional dependency.

In [ ]:
import tkinter as tk
from tkinter import ttk
from pathlib import Path

root = tk.Tk()
root.title("Image reference pattern")
asset_path = Path("assets") / "logo.png"  # Replace with an existing image.
try:
    logo = tk.PhotoImage(file=str(asset_path))
except tk.TclError as exc:
    ttk.Label(root, text=f"Could not load image: {exc}", padding=16).pack()
else:
    root.logo_image = logo  # Keep a long-lived Python reference.
    ttk.Label(root, image=root.logo_image).pack(padx=16, pady=16)
root.mainloop()

## 3. Custom drawings versus analytical plots

For data analytics, Canvas is appropriate for lightweight schematic visualizations such as a progress indicator, a small hand-drawn distribution sketch, or an annotated process diagram. For axes, scales, legends, multiple data series, statistical transforms, and publication-quality figures, use Matplotlib or another charting library and embed its rendering in the Tk application.

Keep the data transformation separate from drawing code. Canvas should receive prepared coordinates or summaries, not query a database or clean a DataFrame on every redraw. This makes it easier to resize the view and to test the calculations independently of the GUI.

## Guided lab: Lab: resize-aware annotation

Build a Canvas visualization that displays monthly sales bars and labels. It must redraw when the window resizes, handle an empty series, avoid division by zero, and explain units. Add a selected-bar highlight or tooltip. Compare the result with an equivalent Matplotlib chart and write down which approach is preferable for the capstone and why.

## Review questions and practice

1. Why can a Tk PhotoImage disappear after it is assigned to a local variable?
2. What does `canvas.delete("all")` do in a redraw routine?
3. Why is Canvas less suitable than Matplotlib for a complex statistical chart?
4. What should a redraw function do when all values are zero?

## Further reading

- Canvas: https://docs.python.org/3/library/tkinter.html#the-canvas-widget
- Pillow: https://pillow.readthedocs.io/

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.